# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library, referencing dataset elements by their `@id` as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata (this will download and interpret the schema)
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Examine available record sets and their schema definitions
record_sets = list(metadata.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets):
    print(f"Record Set {i+1} - Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    # Show field info for each record set
    fields = getattr(record_set, 'fields', [])
    print(f"  Fields in record set:")
    for field in fields:
        print(f"    - {field.name} (@id: {field.id}, datatype: {field.data_type})")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.
All references use `@id` as required for reproducibility.

In [ ]:
# Collect available record set `@id`s for extraction
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for rset_id in record_set_ids:
    # Load records for each record set by its @id
    records = list(dataset.records(record_set=rset_id))
    # If records are present, convert to DataFrame
    if records:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded {len(df)} rows for Record Set: {rset_id}")

# Show columns of the first populated dataframe (if any)
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set {first_id}:")
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())
else:
    print("No records found in the dataset record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records by a numeric field, normalizing, and grouping.
Below you must use the field `@id` to refer to columns. Adjust numeric_field and group_field as appropriate for the dataset you loaded above.

In [ ]:
# Example: Select a numeric field and group field using their `@id`

# (You may need to inspect the output of previous cells to tailor these if the dataset changes)
if dataframes:
    active_id = list(dataframes.keys())[0]
    df = dataframes[active_id]
    print(f"Examining DataFrame for record set: {active_id}")

    # Attempt to auto-select a numeric column (float/int)
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if len(numeric_cols) == 0:
        print("No numeric fields found for EDA.")
    else:
        numeric_field = numeric_cols[0]  # Use the first numeric column for demonstration
        print(f"Using numeric field '@id': {numeric_field}")

        # Simple filtering: values > threshold (for demonstration)
        threshold = df[numeric_field].mean() if df[numeric_field].dtype in ['float64','int64'] else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}: (Showing up to 5 rows)")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1)
        )
        print(f"Normalized {numeric_field} (first 5 rows):")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Find categorical/text columns for grouping
        group_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = group_fields[0] if group_fields else None

        if group_field:
            print(f"Grouping by field '@id': {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped.head())
        else:
            print("No suitable group (categorical) fields found for grouping.")
else:
    print("No DataFrame to perform EDA on.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib and pandas plotting functions.
The plot below demonstrates a histogram and a boxplot for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and ('numeric_field' in locals() and numeric_field in df.columns):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Histogram for {numeric_field}")
    plt.xlabel(numeric_field)

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot for {numeric_field}")
    plt.xlabel(numeric_field)

    plt.tight_layout()
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we have:
- Loaded the FAIR^2 dataset via its Croissant schema and explored its metadata.
- Inspected the structure of available record sets, fields, and their `@id`s, referencing everything by `@id`.
- Extracted data into pandas DataFrames and demonstrated filtering, normalization, and groupby-mean operations on a numeric field.
- Created distribution visualizations for the selected numeric field.

**Next Steps:**
You may continue with domain-specific analysis and modeling, using field and record set identifiers as found in the Croissant schema for reproducible, FAIR data analysis.